In [1]:
from datasets import load_dataset
import random
from tqdm import tqdm
import os
import json
import hashlib

In [2]:
def text_to_id(text):
    """Return a deterministic ID for the given text."""
    # Normalize whitespace etc. to avoid accidental differences
    return hashlib.md5(text.encode("utf-8")).hexdigest()

In [3]:
# ds = load_dataset("hotpotqa/hotpot_qa", "distractor")
ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

README.md: 0.00B [00:00, ?B/s]

c:\Users\deevy\anaconda3\envs\cs6101_assgn1\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\deevy\.cache\huggingface\hub\datasets--hotpotqa--hotpot_qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular

train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

In [4]:
num_bad_docs = 2

In [5]:
random.seed(42) 

skip_count = 0
train_data = []
for row in tqdm(ds['train'], total=len(ds['train'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                train_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(train_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 90447/90447 [00:14<00:00, 6312.59it/s]

Skipped 418 samples due to insufficient bad documents.


In [6]:
random.seed(42) 

skip_count = 0
validation_data = []
for row in tqdm(ds['validation'], total=len(ds['validation'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                validation_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(validation_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 7405/7405 [00:01<00:00, 6290.06it/s]

Skipped 28 samples due to insufficient bad documents.


In [7]:
len(train_data), len(validation_data)

(360116, 29508)

In [8]:
os.makedirs("data", exist_ok=True)
with open("data/train_data.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")
with open("data/validation_data.jsonl", "w") as f:
    for item in validation_data:
        f.write(json.dumps(item) + "\n")

In [9]:
os.makedirs("data/docs", exist_ok=True)
doc_ids = set()
with open("data/docs/doc_data.jsonl", "w") as f:
    for item in train_data:
        q = item["query"]
        q_id = text_to_id(q)
        if q_id not in doc_ids:
            doc_ids.add(q_id)
            f.write(json.dumps({"id": q_id, "contents": q}) + "\n")
        gd = item["good_doc"]
        gd_id = text_to_id(gd)
        if gd_id not in doc_ids:
            doc_ids.add(gd_id)
            f.write(json.dumps({"id": gd_id, "contents": gd}) + "\n")
        bd = item["bad_doc"]
        bd_id = text_to_id(bd)
        if bd_id not in doc_ids:
            doc_ids.add(bd_id)
            f.write(json.dumps({"id": bd_id, "contents": bd}) + "\n")
    for item in validation_data:
        q = item["query"]
        q_id = text_to_id(q)
        if q_id not in doc_ids:
            doc_ids.add(q_id)
            f.write(json.dumps({"id": q_id, "contents": q}) + "\n")
        gd = item["good_doc"]
        gd_id = text_to_id(gd)
        if gd_id not in doc_ids:
            doc_ids.add(gd_id)
            f.write(json.dumps({"id": gd_id, "contents": gd}) + "\n")
        bd = item["bad_doc"]
        bd_id = text_to_id(bd)
        if bd_id not in doc_ids:
            doc_ids.add(bd_id)
            f.write(json.dumps({"id": bd_id, "contents": bd}) + "\n")
print(f"Wrote {len(doc_ids)} unique documents.")

Wrote 415838 unique documents.
